In [26]:
# Install dependencies as needed:
%pip install "kagglehub[pandas-datasets]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns # For creating plots

In [11]:
from pathlib import Path
import kagglehub

csv_files = list(Path.cwd().rglob("Telco-Customer-Churn.csv"))

if not csv_files:

    dataset_path = Path(
        kagglehub.dataset_download("blastchar/telco-customer-churn")
    )
    csv_files = list(dataset_path.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Could not find the Telco churn CSV file.")

data = pd.read_csv(csv_files[0])
print(f"Loaded: {csv_files[0]}")

Loaded: C:\Users\HomePC\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1\WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
data.TotalCharges = pd.to_numeric(data.TotalCharges, errors='coerce')
data.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [ ]:
#Removing missing values 
data.dropna(inplace = True)

In [ ]:
# data.isnull().sum()

In [ ]:
# check duplicates
data.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Length: 7032, dtype: bool

## Part 3


## Part 4 - Train/Validation/Test Split

In [17]:
from pathlib import Path
import kagglehub
from sklearn.model_selection import train_test_split

# Separate features and target
if "data" not in globals():
    csv_files = list(Path.cwd().rglob("Telco-Customer-Churn.csv"))
    if not csv_files:
        dataset_path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
        csv_files = list(dataset_path.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("Could not find or download the Telco churn CSV file.")
    data = pd.read_csv(csv_files[0])

data = data.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data.dropna(inplace=True)

df = data.copy()
X = df.drop(columns=["Churn", "customerID"], errors="ignore")
y = df["Churn"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [18]:
# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [ ]:
# Check the sizes
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

## Part 6 - Evaluation and Error Analysis

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, (y_train == "Yes").astype(int))
y_val_binary = (y_val == "Yes").astype(int)
y_test_binary = (y_test == "Yes").astype(int)

val_predictions = model.predict(X_val)
test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test_binary, test_predictions),
    "precision": precision_score(y_test_binary, test_predictions),
    "recall": recall_score(y_test_binary, test_predictions),
    "f1": f1_score(y_test_binary, test_predictions),
    "roc_auc": roc_auc_score(y_test_binary, test_probabilities),
}
print("Test metrics:")
print(pd.Series(metrics).round(3))
print("\nClassification report:")
print(classification_report(y_test_binary, test_predictions, target_names=["No churn", "Churn"]))
print("Confusion matrix [actual rows, predicted columns]:")
print(confusion_matrix(y_test_binary, test_predictions))

Test metrics:
accuracy     0.724
precision    0.488
recall       0.775
f1           0.599
roc_auc      0.822
dtype: float64

Classification report:
              precision    recall  f1-score   support

    No churn       0.90      0.71      0.79       775
       Churn       0.49      0.78      0.60       280

    accuracy                           0.72      1055
   macro avg       0.69      0.74      0.69      1055
weighted avg       0.79      0.72      0.74      1055

Confusion matrix [actual rows, predicted columns]:
[[547 228]
 [ 63 217]]


### Error Analysis

In [21]:
error_analysis = X_test.copy()
error_analysis["actual_churn"] = y_test_binary.to_numpy()
error_analysis["predicted_churn"] = test_predictions
error_analysis["churn_probability"] = test_probabilities
error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual_churn"] == 1) & (error_analysis["predicted_churn"] == 0),
        (error_analysis["actual_churn"] == 0) & (error_analysis["predicted_churn"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())
print("\nError rates by contract type:")
print(pd.crosstab(
    error_analysis["Contract"],
    error_analysis["error_type"],
    normalize="index",
).round(3))

false_negatives = error_analysis[error_analysis["error_type"] == "false_negative"]
print("\nHighest-risk false negatives:")
print(false_negatives.sort_values("churn_probability").head(10))

Error counts:
error_type
correct           764
false_positive    228
false_negative     63
Name: count, dtype: int64

Error rates by contract type:
error_type      correct  false_negative  false_positive
Contract                                               
Month-to-month    0.571           0.061           0.368
One year          0.827           0.102           0.071
Two year          0.980           0.020           0.000

Highest-risk false negatives:
      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
4513  Female              1     Yes        Yes      72          Yes   
2894  Female              0      No         No      48          Yes   
6869  Female              0     Yes        Yes      45          Yes   
3834  Female              0     Yes        Yes      57          Yes   
320   Female              1      No         No      54          Yes   
6038  Female              0     Yes        Yes      70          Yes   
4698  Female              1     Yes         

## Part 7 - Deployment: Batch Scoring

In [23]:
def batch_score(input_frame, fitted_model=model):
    """Return churn predictions and probabilities for a customer batch."""
    features = input_frame.drop(columns=["Churn"], errors="ignore")
    scored = input_frame.copy()
    scored["churn_probability"] = fitted_model.predict_proba(features)[:, 1]
    scored["predicted_churn"] = np.where(
        scored["churn_probability"] >= 0.5,
        "Yes",
        "No",
    )
    return scored

# Example batch job: replace X_test with a newly received customer file.
scored_customers = batch_score(X_test)
print(scored_customers[["churn_probability", "predicted_churn"]].head())
# scored_customers.to_csv("telco_churn_scored.csv", index=False)

      churn_probability predicted_churn
2778           0.547713             Yes
715            0.587862             Yes
153            0.035261              No
3755           0.879616             Yes
4848           0.890907             Yes


## Part 8 - Monitoring: Quality and Data Drift

In [25]:
def population_stability_index(reference, current, bins=10):
    """Calculate PSI; values above 0.20 are commonly treated as material drift."""
    edges = np.unique(np.quantile(reference.dropna(), np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        return 0.0
    edges[0] = -np.inf
    edges[-1] = np.inf
    reference_distribution = pd.cut(reference, bins=edges, include_lowest=True).value_counts(normalize=True)
    current_distribution = pd.cut(current, bins=edges, include_lowest=True).value_counts(normalize=True)
    reference_distribution, current_distribution = reference_distribution.align(
        current_distribution,
        fill_value=0,
    )
    epsilon = 1e-6
    return float(((current_distribution + epsilon - reference_distribution - epsilon)
                  * np.log((current_distribution + epsilon) / (reference_distribution + epsilon))).sum())


def categorical_drift(reference, current):
    reference_distribution = reference.fillna("<missing>").value_counts(normalize=True)
    current_distribution = current.fillna("<missing>").value_counts(normalize=True)
    reference_distribution, current_distribution = reference_distribution.align(
        current_distribution,
        fill_value=0,
    )
    return float((current_distribution - reference_distribution).abs().sum())


def monitor_drift(reference_frame, current_frame):
    drift = {}
    for column in reference_frame.columns:
        if column not in current_frame:
            continue
        if pd.api.types.is_numeric_dtype(reference_frame[column]):
            drift[column] = population_stability_index(
                reference_frame[column],
                current_frame[column],
            )
        else:
            drift[column] = categorical_drift(
                reference_frame[column],
                current_frame[column],
            )
    return pd.Series(drift).sort_values(ascending=False)


def monitor_quality(actual_labels, predicted_labels, predicted_probabilities):
    actual_binary = (pd.Series(actual_labels) == "Yes").astype(int)
    predicted_binary = (pd.Series(predicted_labels) == "Yes").astype(int)
    return pd.Series({
        "accuracy": accuracy_score(actual_binary, predicted_binary),
        "f1": f1_score(actual_binary, predicted_binary),
        "roc_auc": roc_auc_score(actual_binary, predicted_probabilities),
    })

print("Current test-set quality:")
print(monitor_quality(y_test, np.where(test_predictions == 1, "Yes", "No"), test_probabilities).round(3))
print("\nDrift against the training reference (PSI for numeric, L1 distance for categorical):")
print(monitor_drift(X_train, X_test).round(3))
# Alert when a numeric PSI exceeds 0.20 or a categorical L1 distance exceeds 0.20.


Current test-set quality:
accuracy    0.724
f1          0.599
roc_auc     0.822
dtype: float64

Drift against the training reference (PSI for numeric, L1 distance for categorical):
gender              0.051
PaymentMethod       0.050
StreamingTV         0.033
TechSupport         0.032
Partner             0.030
PhoneService        0.029
MultipleLines       0.029
OnlineBackup        0.026
StreamingMovies     0.025
DeviceProtection    0.019
InternetService     0.017
OnlineSecurity      0.017
Contract            0.017
MonthlyCharges      0.013
PaperlessBilling    0.009
Dependents          0.005
tenure              0.005
TotalCharges        0.004
SeniorCitizen       0.000
dtype: float64
